# 11 · Pitt: las 8 emociones de Mikels, positivas y negativas (CLIP congelado + sonda, solo imagen)

**Pregunta:** ¿se reconoce desde la imagen **qué emoción** provoca un anuncio (no solo su valencia)? ¿Qué tan bien se reconoce cada una?

**Etiquetas.** Pitt no usa las emociones de Mikels: se derivan de sus 30 sentimientos. El modelo **solo ve la imagen**; las anotaciones solo definen la respuesta correcta.

| Valencia | Emoción de Mikels (EmoSet) | Sentimientos de Pitt |
|---|---|---|
| Positiva | amusement | amused (humored, laughing) |
| Positiva | awe | amazed (surprised, astonished, awed, fascinated, intrigued) |
| Positiva | contentment | calm (soothed, peaceful, cozy), cheerful (happy, joyful), grateful, loving |
| Positiva | excitement | active (energetic, adventurous, enthusiastic), eager (passionate, hungry, thirsty) |
| Negativa | anger | angry (annoyed, irritated) |
| Negativa | disgust | disturbed (disgusted, shocked) |
| Negativa | fear | afraid (horrified, scared), alarmed (concerned, worried, anxious) |
| Negativa | sadness | sad (depressed, upset, betrayed) |
| — | *(sin equivalente)* | confident, inspired, persuaded, proud, jealous, pessimistic y los neutros |

**Emoción de cada anotador:** la de Mikels con más sentimientos marcados en su voto (si empata o no marcó ninguna, no vota).

**Dos análisis:**
- **A · Clasificación (emoción de la mayoría).** El anuncio recibe la emoción elegida por **más de la mitad** de sus anotadores. Las negativas casi nunca alcanzan mayoría (los anotadores rara vez coinciden en *cuál* emoción negativa): solo se evalúan las emociones con al menos 15 anuncios.
- **B · Detección por emoción (al menos un anotador).** Para cada una de las 8 emociones: ¿el modelo distingue los anuncios en que **algún** anotador la vio de los demás? Se mide con ROC-AUC (0,5 = azar). Permite evaluar las 4 negativas.

**Tres formas de usar CLIP congelado:**
- **EmoSet → Pitt:** sonda entrenada en las 8 emociones de EmoSet (embeddings de la etapa 1 del notebook 09), aplicada a Pitt **sin ver ningún anuncio**.
- **Sonda Pitt:** entrenada en Pitt con validación cruzada estratificada (pesos de clase balanceados).
- **Zero-shot:** frases por emoción, sin entrenamiento.

**Requisito:** etapa 1 del notebook 09 ejecutada (`resultados/emoset_modelos/emb_emoset_clip.npz` y `emoset/emoset_index.csv` en Drive).

## 1. Entorno y rutas

In [ ]:
import glob
import io
import json
import os
import random
import re
import time
import urllib.request
import zipfile
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score,
                             precision_recall_fscore_support, roc_auc_score)
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.isdir('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/publicity_analysis'
    PITT_DIR = os.path.join(DRIVE_ROOT, 'pitt')
    IMG_DIR = '/content/work/pitt_256'
    PITT_CACHE_ZIP = os.path.join(PITT_DIR, 'etapa2_256.zip')
    OUT = os.path.join(DRIVE_ROOT, 'resultados', 'pitt_mikels')
else:
    # Google Drive para escritorio: se busca la cuenta que tiene los resultados de la etapa 1
    cands = [c for d in ('Mi unidad', 'My Drive')
             for c in glob.glob(os.path.expanduser(f'~/Library/CloudStorage/GoogleDrive-*/{d}/publicity_analysis'))]
    DRIVE_ROOT = next(c for c in cands if os.path.exists(os.path.join(c, 'resultados', 'emoset_modelos', 'emb_emoset_clip.npz')))
    PITT_DIR = 'pitt'
    IMG_DIR = os.path.expanduser('~/datasets/pitt_256')
    PITT_CACHE_ZIP = None
    OUT = os.path.join('resultados', 'pitt_mikels')
VOTOS_IMG = os.path.join(PITT_DIR, 'votos', 'image')
EMB_EMOSET = os.path.join(DRIVE_ROOT, 'resultados', 'emoset_modelos', 'emb_emoset_clip.npz')
EMOSET_INDEX = os.path.join(DRIVE_ROOT, 'emoset', 'emoset_index.csv')
for d in (IMG_DIR, OUT):
    os.makedirs(d, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
CLIP_NAME = 'openai/clip-vit-base-patch16'
URL_VOTOS = 'https://people.cs.pitt.edu/~kovashka/ads/annotations_images.zip'
URL_IMG = 'https://people.cs.pitt.edu/~mzhang/image_ads/'
UA = {'User-Agent': 'Mozilla/5.0 (research dataset downloader)'}
SEED = 0
N_EXCITEMENT = 1000     # análisis A: excitement es mayoritaria, se toma una muestra; el resto se usa completo
MIN_N = 15              # análisis A: emociones con menos anuncios no se evalúan
N_ALEATORIA = 4000      # análisis B: anuncios al azar (más todos los que tienen alguna emoción negativa)
N_SPLITS = 5
MIKELS = ['amusement', 'awe', 'contentment', 'excitement', 'anger', 'disgust', 'fear', 'sadness']
POSITIVAS = MIKELS[:4]
COL_EMO = {e: ('#4a78b5' if e in POSITIVAS else '#c0504d') for e in MIKELS}
pd.set_option('display.float_format', '{:.3f}'.format)
print('Dispositivo:', DEVICE, '· resultados en', OUT)

## 2. Etiquetas: emociones de Mikels por anuncio

In [ ]:
def fetch(url, timeout=60, retries=3):
    for k in range(retries):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=timeout) as r:
                return r.read()
        except Exception:
            if k == retries - 1:
                raise
            time.sleep(1.5 * (k + 1))


if not os.path.exists(os.path.join(VOTOS_IMG, 'Sentiments.json')):
    with zipfile.ZipFile(io.BytesIO(fetch(URL_VOTOS))) as z:
        z.extractall(os.path.join(PITT_DIR, 'votos'))

SENT_NAMES = {}
for line in open(os.path.join(VOTOS_IMG, 'Sentiments_List.txt'), encoding='latin-1'):
    m = re.match(r'\s*(\d+)\.?\s+"[^"]+".*ABBREVIATION:\s*"([^"]+)"', line)
    if m:
        SENT_NAMES[m.group(1)] = m.group(2)

SENT_MIKELS = {'amused': 'amusement', 'amazed': 'awe',
               'calm': 'contentment', 'cheerful': 'contentment', 'grateful': 'contentment', 'loving': 'contentment',
               'active': 'excitement', 'eager': 'excitement',
               'angry': 'anger', 'disturbed': 'disgust', 'afraid': 'fear', 'alarmed': 'fear', 'sad': 'sadness'}


def emocion_anotador(ids):
    c = Counter(SENT_MIKELS[SENT_NAMES.get(str(i))] for i in ids if SENT_NAMES.get(str(i)) in SENT_MIKELS)
    top = c.most_common(2)
    if not top or (len(top) == 2 and top[0][1] == top[1][1]):
        return None
    return top[0][0]


filas = []
for key, vs in json.load(open(os.path.join(VOTOS_IMG, 'Sentiments.json'))).items():
    if len(vs) < 2:
        continue
    e = [emocion_anotador(v) for v in vs]
    c = Counter(x for x in e if x)
    emo, k = c.most_common(1)[0] if c else (None, 0)
    filas.append({'key': key, 'n_anotadores': len(vs),
                  'emocion': emo if k / len(vs) > 0.5 else None, 'acuerdo': k / len(vs),
                  **{f'votos_{m}': c.get(m, 0) for m in MIKELS}})
pitt = pd.DataFrame(filas)
conteo = pd.DataFrame({
    'valencia': ['positiva' if m in POSITIVAS else 'negativa' for m in MIKELS],
    'mayoría (A)': [int((pitt['emocion'] == m).sum()) for m in MIKELS],
    '≥ 2 anotadores': [int((pitt[f'votos_{m}'] >= 2).sum()) for m in MIKELS],
    '≥ 1 anotador (B)': [int((pitt[f'votos_{m}'] >= 1).sum()) for m in MIKELS],
}, index=MIKELS)
print(f'{len(pitt)} anuncios con votos')
display(conteo)

## 3. Muestras de los dos análisis, imágenes y embeddings

- **A:** emociones con al menos `MIN_N` anuncios por mayoría; excitement se submuestrea a 1.000.
- **B:** 4.000 anuncios al azar + todos los anuncios en que algún anotador vio una emoción negativa (para tener suficientes ejemplos de anger, disgust y sadness).

Embeddings con el **mismo preprocesamiento** de EmoSet en el notebook 09 (lado menor 256 → recorte central 224), normalizados.

In [ ]:
EMO_A = [m for m in MIKELS if conteo.loc[m, 'mayoría (A)'] >= MIN_N]
print('Análisis A, emociones evaluables:', EMO_A, '· excluidas:', [m for m in MIKELS if m not in EMO_A])
partes = []
for m in EMO_A:
    g = pitt[pitt['emocion'] == m]
    partes.append(g.sample(n=N_EXCITEMENT, random_state=SEED) if m == 'excitement' and len(g) > N_EXCITEMENT else g)
data_a = pd.concat(partes).sort_values('key').reset_index(drop=True)

keys = pitt['key'].tolist()
random.Random(SEED).shuffle(keys)
con_neg = pitt.loc[pitt[[f'votos_{m}' for m in MIKELS[4:]]].sum(1) > 0, 'key']
data_b = pitt[pitt['key'].isin(set(keys[:N_ALEATORIA]) | set(con_neg))].sort_values('key').reset_index(drop=True)
print(f'A: {len(data_a)} anuncios ·', data_a['emocion'].value_counts().to_dict())
print(f'B: {len(data_b)} anuncios')

local_name = lambda key: key.replace('/', '_').rsplit('.', 1)[0] + '.jpg'
if PITT_CACHE_ZIP and os.path.exists(PITT_CACHE_ZIP):
    with zipfile.ZipFile(PITT_CACHE_ZIP) as z:
        z.extractall(IMG_DIR)


def bajar(key):
    dst = os.path.join(IMG_DIR, local_name(key))
    if os.path.exists(dst):
        return 'existe'
    try:
        im = Image.open(io.BytesIO(fetch(URL_IMG + key))).convert('RGB')
        s = 256 / min(im.size)
        if s < 1:
            im = im.resize((round(im.width * s), round(im.height * s)), Image.BICUBIC)
        im.save(dst + '.part', 'JPEG', quality=92)
        os.replace(dst + '.part', dst)
        return 'nueva'
    except Exception:
        return 'error'


todos = sorted(set(data_a['key']) | set(data_b['key']))
t0 = time.time()
with ThreadPoolExecutor(max_workers=16) as ex:
    estados = Counter(ex.map(bajar, todos))
print('Imágenes:', dict(estados), f'· {time.time() - t0:.0f}s')

from transformers import CLIPTextModelWithProjection, CLIPTokenizer, CLIPVisionModelWithProjection
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(),
                T.Normalize([0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711])])
l2 = lambda X: X / np.linalg.norm(X, axis=1, keepdims=True)

emb_path = os.path.join(OUT, 'clip_img_emb_pitt_8emo.npz')
EMB = {}
if os.path.exists(emb_path):
    d = np.load(emb_path)
    EMB = dict(zip(d['key'], d['X']))
faltan = [k for k in todos if k not in EMB and os.path.exists(os.path.join(IMG_DIR, local_name(k)))]
if faltan:
    vis = CLIPVisionModelWithProjection.from_pretrained(CLIP_NAME).to(DEVICE).eval()
    t0 = time.time()
    with torch.no_grad():
        for i in range(0, len(faltan), 64):
            ks = faltan[i:i + 64]
            x = torch.stack([tf(Image.open(os.path.join(IMG_DIR, local_name(k))).convert('RGB')) for k in ks]).to(DEVICE)
            EMB.update(zip(ks, vis(pixel_values=x).image_embeds.float().cpu().numpy()))
    np.savez(emb_path, key=np.array(list(EMB), dtype=str), X=np.stack(list(EMB.values())))
    print(f'{len(faltan)} embeddings nuevos en {time.time() - t0:.0f}s')
data_a = data_a[data_a['key'].isin(EMB)].reset_index(drop=True)
data_b = data_b[data_b['key'].isin(EMB)].reset_index(drop=True)
XA = l2(np.stack([EMB[k] for k in data_a['key']]).astype(np.float32))
XB = l2(np.stack([EMB[k] for k in data_b['key']]).astype(np.float32))
print('XA', XA.shape, '· XB', XB.shape)

## 4. Los tres modelos

- **Sonda EmoSet (8 emociones):** regresión logística sobre los embeddings de CLIP del *train* de EmoSet; `C` elegido en *val*. Control: su rendimiento en el *test* de EmoSet.
- **Zero-shot:** las mismas frases del notebook 09 ("a photo that evokes fear.", …), promediadas por emoción.

In [ ]:
idx = pd.read_csv(EMOSET_INDEX, usecols=['split', 'emotion'])
Z = np.load(EMB_EMOSET)
EMO = {}
for s in ('train', 'val', 'test'):
    e = idx.loc[idx['split'] == s, 'emotion'].to_numpy()
    assert len(e) == len(Z[s]), 'El índice de EmoSet no coincide con los embeddings'
    EMO[s] = (l2(Z[s].astype(np.float32)), np.array([MIKELS.index(x) for x in e]))

t0 = time.time()
best = (-1, None, None)
for C in (0.01, 0.1, 1.0):
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000)).fit(*EMO['train'])
    f1 = f1_score(EMO['val'][1], clf.predict(EMO['val'][0]), average='macro')
    if f1 > best[0]:
        best = (f1, C, clf)
clf_emoset = best[2]
assert list(clf_emoset.classes_) == list(range(8))
pred_emoset_test = clf_emoset.predict(EMO['test'][0])
print(f'Sonda EmoSet: C={best[1]} · test EmoSet (8 emociones): acc {accuracy_score(EMO["test"][1], pred_emoset_test):.3f} · '
      f'macro-F1 {f1_score(EMO["test"][1], pred_emoset_test, average="macro"):.3f} · {time.time() - t0:.0f}s')

txt = CLIPTextModelWithProjection.from_pretrained(CLIP_NAME).to(DEVICE).eval()
tok = CLIPTokenizer.from_pretrained(CLIP_NAME)
TEMPLATES = ['a photo that evokes {}.', 'an image that makes people feel {}.',
             'a photo expressing {}.', 'a picture that conveys a feeling of {}.']
with torch.no_grad():
    W = []
    for e in MIKELS:
        t = tok([tp.format(e) for tp in TEMPLATES], padding=True, return_tensors='pt').to(DEVICE)
        W.append(F.normalize(F.normalize(txt(**t).text_embeds.float(), dim=-1).mean(0), dim=0).cpu().numpy())
W = np.stack(W)
zs_scores = lambda X: torch.softmax(torch.tensor(100 * X @ W.T), 1).numpy()   # probabilidad por emoción

## 5. Análisis A · Clasificación por emoción de la mayoría

La sonda EmoSet y el zero-shot eligen entre **las 8 emociones** (si predicen una emoción no evaluable, cuenta como error). La sonda Pitt elige entre las emociones evaluables (validación cruzada 5 × 5). Acierto flexible: la predicción coincide con la emoción de **algún** anotador.

In [ ]:
y_a = data_a['emocion'].map(MIKELS.index).to_numpy()
lab_a = [MIKELS.index(m) for m in EMO_A]

PRED = {'sonda EmoSet → Pitt': clf_emoset.predict(XA)[:, None],
        'zero-shot': zs_scores(XA).argmax(1)[:, None]}
t0 = time.time()
cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=5, random_state=SEED)
PB = np.zeros((len(y_a), 5), int)
for k, (tr, te) in enumerate(cv.split(XA, y_a)):
    gs = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=3000)),
                      {'logisticregression__C': [1e-3, 1e-2, 1e-1, 1.0]}, scoring='f1_macro',
                      cv=StratifiedKFold(3, shuffle=True, random_state=k)).fit(XA[tr], y_a[tr])
    PB[te, k // N_SPLITS] = gs.predict(XA[te])
PRED['sonda Pitt (CV)'] = PB
print(f'Sonda Pitt: {time.time() - t0:.0f}s')

votos_a = data_a[[f'votos_{m}' for m in MIKELS]].to_numpy()


def por_emocion(y, P):
    out = [precision_recall_fscore_support(y, P[:, j], labels=lab_a, zero_division=0) for j in range(P.shape[1])]
    return [np.mean([o[i] for o in out], 0) for i in range(3)] + [out[0][3]]


g = np.random.default_rng(SEED)
boot = [g.integers(0, len(y_a), len(y_a)) for _ in range(500)]
filas, resumen, CMS = [], [], {}
for nombre, P in PRED.items():
    pr, rc, f1, sup = por_emocion(y_a, P)
    f1_b = np.array([por_emocion(y_a[i], P[i])[2] for i in boot])
    for c, e in enumerate(EMO_A):
        lo, hi = np.percentile(f1_b[:, c], [2.5, 97.5])
        filas.append({'protocolo': nombre, 'emoción': e, 'anuncios': int(sup[c]), 'precisión': pr[c],
                      'recall': rc[c], 'f1': f1[c], 'f1_ic95': f'[{lo:.2f}, {hi:.2f}]'})
    mf = f1_b.mean(1)
    flex = np.mean([[votos_a[i, p] > 0 for p in P[i]] for i in range(len(P))])
    resumen.append({'protocolo': nombre,
                    'accuracy': np.mean([accuracy_score(y_a, P[:, j]) for j in range(P.shape[1])]),
                    'balanced_acc': np.mean([np.mean([(P[y_a == c, j] == c).mean() for c in lab_a]) for j in range(P.shape[1])]),
                    'macro_f1': f1.mean(), 'macro_f1_ic95': f'[{np.percentile(mf, 2.5):.3f}, {np.percentile(mf, 97.5):.3f}]',
                    'acierto_flexible': flex})
    CMS[nombre] = sum(confusion_matrix(y_a, P[:, j], labels=range(8)) for j in range(P.shape[1]))[lab_a] / P.shape[1]

pr, rc, f1, sup = precision_recall_fscore_support(EMO['test'][1], pred_emoset_test, labels=range(8), zero_division=0)
for c, e in enumerate(MIKELS):
    filas.append({'protocolo': 'control · sonda EmoSet en test EmoSet', 'emoción': e, 'anuncios': int(sup[c]),
                  'precisión': pr[c], 'recall': rc[c], 'f1': f1[c]})
por_emo = pd.DataFrame(filas)
res_a = pd.DataFrame(resumen).set_index('protocolo')
por_emo.to_csv(os.path.join(OUT, 'A_clasificacion_por_emocion.csv'), index=False)
res_a.to_csv(os.path.join(OUT, 'A_clasificacion_resumen.csv'))
print('Referencia: siempre "excitement" →', f'accuracy {(y_a == MIKELS.index("excitement")).mean():.3f}',
      f'· macro-F1 {f1_score(y_a, np.full_like(y_a, MIKELS.index("excitement")), average="macro", labels=lab_a):.3f}')
display(res_a)
display(por_emo.set_index(['protocolo', 'emoción']))

In [ ]:
fig = plt.figure(figsize=(18, 9))
ax = fig.add_subplot(2, 1, 1)
prots = list(PRED) + ['control · sonda EmoSet en test EmoSet']
cols = ['#4a78b5', '#9a9a9a', '#c9a227', '#cfcfcf']
w = 0.2
n_a = data_a['emocion'].value_counts()
for k, (p, col) in enumerate(zip(prots, cols)):
    sub = por_emo[por_emo['protocolo'] == p].set_index('emoción').loc[EMO_A]
    x = np.arange(len(EMO_A)) + (k - 1.5) * w
    ax.bar(x, sub['f1'], w, color=col, label=p)
    if sub['f1_ic95'].notna().all():
        ci = sub['f1_ic95'].str.strip('[]').str.split(', ', expand=True).astype(float)
        ax.errorbar(x, sub['f1'], yerr=[sub['f1'] - ci[0], ci[1] - sub['f1']], fmt='none', ecolor='black', capsize=2)
ax.set_xticks(range(len(EMO_A)), [f'{e}\n(n Pitt = {n_a.get(e, 0)})' for e in EMO_A])
for t in ax.get_xticklabels():
    t.set_color(COL_EMO[t.get_text().split('\n')[0]])
ax.set(ylabel='F1 (IC95%)', ylim=(0, 1), title='A · F1 por emoción (mayoría de anotadores) · azul: positivas · rojo: negativas')
ax.legend(fontsize=8, ncol=2)
for k, nombre in enumerate(PRED):
    ax = fig.add_subplot(2, 3, 4 + k)
    cm = CMS[nombre]
    cmn = cm / cm.sum(1, keepdims=True)
    ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1, aspect='auto')
    for i in range(cmn.shape[0]):
        for j in range(8):
            if cmn[i, j] >= 0.005:
                ax.text(j, i, f'{cmn[i, j]:.2f}', ha='center', va='center', fontsize=7,
                        color='white' if cmn[i, j] > 0.5 else 'black')
    ax.set_xticks(range(8), MIKELS, rotation=40, ha='right', fontsize=7)
    ax.set_yticks(range(len(EMO_A)), EMO_A, fontsize=8)
    ax.set(title=nombre, xlabel='predicha', ylabel='real (mayoría)')
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'A_clasificacion.png'), dpi=150); plt.show()

## 6. Análisis B · Detección de cada emoción (al menos un anotador la vio)

Para cada emoción, **positivo** = algún anotador la eligió; **negativo** = ninguno. Puntaje de cada protocolo:
- **sonda EmoSet → Pitt:** probabilidad de esa emoción según la sonda de 8 emociones de EmoSet (sin entrenar con Pitt).
- **sonda Pitt:** una sonda por emoción (uno contra el resto), validación cruzada 5 × 2.
- **zero-shot:** probabilidad de esa emoción entre las 8 frases.

Se reporta ROC-AUC (0,5 = azar), que no depende de la prevalencia de cada emoción.

In [ ]:
P_emoset_b = clf_emoset.predict_proba(XB)
P_zs_b = zs_scores(XB)
filas = []
t0 = time.time()
for c, m in enumerate(MIKELS):
    y = (data_b[f'votos_{m}'] >= 1).to_numpy().astype(int)
    cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=2, random_state=SEED)
    Pp = np.zeros((len(y), 2))
    for k, (tr, te) in enumerate(cv.split(XB, y)):
        gs = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=3000)),
                          {'logisticregression__C': [1e-2, 1e-1, 1.0]}, scoring='roc_auc',
                          cv=StratifiedKFold(3, shuffle=True, random_state=k)).fit(XB[tr], y[tr])
        Pp[te, k // N_SPLITS] = gs.predict_proba(XB[te])[:, 1]
    filas.append({'emoción': m, 'valencia': 'positiva' if m in POSITIVAS else 'negativa',
                  'anuncios_con_emoción': int(y.sum()), 'prevalencia': y.mean(),
                  'auc_sonda_emoset': roc_auc_score(y, P_emoset_b[:, c]),
                  'auc_sonda_pitt': np.mean([roc_auc_score(y, Pp[:, j]) for j in range(2)]),
                  'auc_zero_shot': roc_auc_score(y, P_zs_b[:, c])})
res_b = pd.DataFrame(filas).set_index('emoción')
res_b.to_csv(os.path.join(OUT, 'B_deteccion_por_emocion.csv'))
print(f'{time.time() - t0:.0f}s')
display(res_b)

fig, ax = plt.subplots(figsize=(14, 4.8))
w = 0.27
for k, (col, lab, c) in enumerate([('auc_sonda_emoset', 'sonda EmoSet → Pitt', '#4a78b5'),
                                   ('auc_sonda_pitt', 'sonda Pitt (CV)', '#c9a227'),
                                   ('auc_zero_shot', 'zero-shot', '#9a9a9a')]):
    ax.bar(np.arange(8) + (k - 1) * w, res_b[col], w, color=c, label=lab)
ax.axhline(0.5, color='black', lw=0.8, ls=':')
ax.set_xticks(range(8), [f'{m}\n(n = {n})' for m, n in zip(MIKELS, res_b['anuncios_con_emoción'])])
for t in ax.get_xticklabels():
    t.set_color(COL_EMO[t.get_text().split('\n')[0]])
ax.set(ylim=(0.4, 1), ylabel='ROC-AUC', title='B · ¿Detecta CLIP (solo imagen) los anuncios en que algún anotador vio cada emoción?')
ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'B_deteccion.png'), dpi=150); plt.show()

## 7. Cómo leer los resultados

- **Conteos (sección 2):** si una emoción casi nunca alcanza mayoría, los anotadores no coinciden en ella; eso limita cualquier evaluación de esa emoción como clase.
- **A · Control vs Pitt:** el control dice qué tan reconocible es cada emoción con las etiquetas de EmoSet. La caída en Pitt mide cuánto no se transfiere de fotografías emocionales a anuncios (y del vocabulario de EmoSet al de Pitt).
- **A · Acierto flexible:** si supera mucho al *accuracy*, el modelo suele elegir una emoción que otro anotador sí vio.
- **B:** AUC > 0,7 indica que la imagen contiene información sobre esa emoción aunque los anotadores no coincidan en ella. Compare **sonda EmoSet** (lo que se sabe de emociones en fotografías) con **sonda Pitt** (lo que se aprende de los anuncios mismos).